<a href="https://colab.research.google.com/github/saagar-stha/FreeCodeCamp/blob/main/fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip uninstall -y tensorflow tensorflow-cpu tensorflow-intel keras
# !pip install tensorflow

In [ ]:
# import libraries
import tensorflow as tf
import pandas as pd
from tensorflow import keras
# !pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
import keras
import tensorflow as tf
import numpy as np

# Clear previous session
keras.backend.clear_session()

# Load data
train_df = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'message'])
test_df = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'message'])

train_df['message'] = train_df['message'].fillna('').astype(str)
test_df['message'] = test_df['message'].fillna('').astype(str)

train_labels = (train_df['label'] == 'spam').astype(float).values
test_labels = (test_df['label'] == 'spam').astype(float).values

# Increased parameters for higher sensitivity
VOCAB_SIZE = 2000
MAX_LEN = 150

vectorize_layer = keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorize_layer.adapt(train_df['message'].values)

model = keras.Sequential([
    keras.Input(shape=(1,), dtype=tf.string),
    vectorize_layer,
    keras.layers.Embedding(VOCAB_SIZE, 64, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64, return_sequences=True)),
    keras.layers.GlobalMaxPooling1D(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Retrain with the more robust architecture
history = model.fit(
    train_df['message'].values,
    train_labels,
    epochs=15,
    validation_data=(test_df['message'].values, test_labels),
    batch_size=32,
    verbose=1
)

In [ ]:
def predict_message(pred_text):
  # Using tf.constant to ensure compatible input for Keras 3 / TF 2.21
  input_tensor = tf.constant([pred_text])
  prediction = model.predict(input_tensor, verbose=0)[0][0]

  label = 'spam' if prediction >= 0.5 else 'ham'
  return [float(prediction), label]

pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
